
# Taller de Ciencia de Datos: Modelo de reposicion/priorizacion de SKU en retail

Este notebook esta disenado como una guia **por completar**. Algunas celdas tienen codigo listo y otras tienen `TODO` para que practiques.

**Caso de negocio:** una empresa retail quiere predecir si un SKU debe ser repuesto o priorizado en una tienda.

**Target:** `target_reponer_sku`

- `1`: conviene reponer/priorizar el SKU.
- `0`: no conviene reponer por ahora.

**Objetivo del taller:** integrar entendimiento de negocio, limpieza, EDA, feature engineering, modelos de clasificacion, ensembles, evaluacion, prediccion y comunicacion del beneficio.



## 0. Pregunta de negocio

Antes de programar, responde:

1. Que decision se quiere mejorar?
2. Que significa equivocarse?
3. Que es mas costoso: no reponer un SKU que si debia reponerse, o reponer un SKU que no era necesario?
4. Que area usaria el resultado: abastecimiento, comercial, tienda, planeamiento o todas?

**Actividad:** escribe tu respuesta en una celda Markdown debajo.


In [ ]:
# Respuesta del analista / estudiante:
# TODO: escribe aqui la decision de negocio que resolvera el modelo.



## 1. Configuracion del entorno

Ejecuta esta celda. Si te falta alguna libreria, instalala con `pip install nombre_libreria`.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.inspection import permutation_importance

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)



## 2. Cargar datos

Coloca el CSV en la misma carpeta del notebook. Si estas trabajando en ChatGPT, el path `/mnt/data/...` ya apunta al archivo generado.


In [ ]:
DATA_PATH = Path('retail_sku_dataset_sintetico_10000_filas_70_variables.csv')

# Fallback para entorno ChatGPT /mnt/data
if not DATA_PATH.exists():
    DATA_PATH = Path('/mnt/data/retail_sku_dataset_sintetico_10000_filas_70_variables.csv')

df_raw = pd.read_csv(DATA_PATH)
print(df_raw.shape)
df_raw.head()



## 3. Entendimiento inicial de variables

Queremos responder:

- Cuantas filas y columnas hay?
- Cual es la variable target?
- Cuales son variables numericas, categoricas y de fecha?
- Cuales tienen nulos, textos inconsistentes o valores invalidos?


In [ ]:
TARGET = 'target_reponer_sku'
ID_COLS = ['sku_id', 'tienda_id']
DATE_COL = 'fecha'

# TODO 1: valida que TARGET exista en el dataframe.
# Pista: usa "TARGET in df_raw.columns"

# TODO 2: calcula la distribucion del target.
# Pista: df_raw[TARGET].value_counts(normalize=True)

# Escribe tu codigo debajo:


In [ ]:
# Resumen rapido de tipos y nulos
resumen_calidad = pd.DataFrame({
    'tipo': df_raw.dtypes.astype(str),
    'nulos': df_raw.isna().sum(),
    'nulos_pct': (df_raw.isna().mean() * 100).round(2),
    'valores_unicos': df_raw.nunique(dropna=True)
}).sort_values('nulos_pct', ascending=False)

resumen_calidad.head(20)



## 4. Diccionario analitico de variables

Clasifica las variables por familia. Esto ayuda a entender el negocio antes de modelar.

**Actividad:** completa listas si encuentras mas variables.


In [ ]:
cols_producto_tienda = [
    'region', 'ciudad', 'distrito', 'formato_tienda', 'canal_venta',
    'cluster_tienda', 'segmento_cliente_principal', 'categoria', 'subcategoria',
    'marca', 'proveedor', 'sku_estado'
]

cols_precio_margen = [
    'precio_lista', 'precio_venta', 'costo_unitario', 'margen_pct',
    'descuento_pct', 'competencia_precio_prom', 'precio_vs_competencia_pct',
    'elasticidad_precio'
]

cols_inventario_ventas = [
    'stock_actual', 'stock_seguridad', 'inventario_transito', 'dias_inventario',
    'ventas_unidades_7d', 'ventas_unidades_30d', 'ventas_valor_30d',
    'unidades_devueltas_30d', 'rotacion_30d', 'quiebres_7d', 'quiebres_30d'
]

cols_forecast = ['forecast_7d', 'forecast_30d', 'error_forecast_pct']

cols_promo_trade = [
    'promo_activa', 'tipo_promo', 'dias_promo_mes', 'inversion_trade',
    'exhibicion_secundaria', 'espacio_gondola_cm', 'puntaje_planograma',
    'prioridad_comercial'
]

cols_operacion_contexto = [
    'cumplimiento_sla_pct', 'fill_rate_pct', 'lead_time_dias',
    'frecuencia_reposicion_dias', 'temperatura_promedio', 'lluvia_mm_mes',
    'feriado_mes', 'evento_local', 'campania_marketing', 'canal_abastecimiento',
    'nivel_socioeconomico_zona', 'distancia_cd_km', 'capacidad_bodega_m3',
    'merma_pct', 'vencimiento_dias', 'vida_util_dias', 'reclamos_30d',
    'rating_cliente', 'score_tendencia_categoria', 'riesgo_quiebre_score'
]

# TODO: revisa si alguna variable falta o sobra en estas familias.



## 5. Limpieza de datos

El dataset viene sucio intencionalmente. Debemos corregir:

- Numeros como texto: `S/ 12.50`, `15%`, `bad_value`.
- Fechas invalidas.
- Categorias con espacios o formatos inconsistentes.
- Valores negativos imposibles.
- Duplicados por `sku_id`, `tienda_id`, `fecha`.


In [ ]:
def parse_numeric_series(s: pd.Series) -> pd.Series:
    # Convierte una serie con simbolos como S/, %, comas y textos invalidos a numerica.
    return pd.to_numeric(
        s.astype(str)
         .str.replace('S/', '', regex=False)
         .str.replace('%', '', regex=False)
         .str.replace(',', '.', regex=False)
         .str.strip()
         .replace({'nan': np.nan, 'None': np.nan, 'N/A': np.nan, 'bad_value': np.nan, '': np.nan}),
        errors='coerce'
    )


def normalize_text_series(s: pd.Series) -> pd.Series:
    # Normaliza textos categoricos.
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r'\s+', ' ', regex=True)
         .replace({'nan': np.nan, 'None': np.nan, 'N/A': np.nan, '': np.nan})
    )


In [ ]:
df = df_raw.copy()

# 5.1 Fecha
# TODO: convierte la columna fecha a datetime con errors='coerce'.
# df[DATE_COL] = ...

# 5.2 Columnas numericas candidatas
numeric_candidates = cols_precio_margen + cols_inventario_ventas + cols_forecast + cols_promo_trade + cols_operacion_contexto
numeric_candidates = [c for c in numeric_candidates if c in df.columns]

# Algunas columnas de esas familias son categoricas. Las sacamos manualmente.
not_numeric = ['tipo_promo', 'prioridad_comercial', 'evento_local', 'campania_marketing', 'canal_abastecimiento', 'nivel_socioeconomico_zona']
numeric_cols = [c for c in numeric_candidates if c not in not_numeric]

for col in numeric_cols:
    df[col] = parse_numeric_series(df[col])

# 5.3 Columnas categoricas
categorical_cols = [c for c in df.columns if c not in numeric_cols + [TARGET, DATE_COL]]
for col in categorical_cols:
    df[col] = normalize_text_series(df[col])

# 5.4 Valores negativos imposibles
cols_no_negativas = [
    'precio_lista', 'precio_venta', 'costo_unitario', 'stock_actual', 'stock_seguridad',
    'inventario_transito', 'ventas_unidades_7d', 'ventas_unidades_30d', 'ventas_valor_30d',
    'forecast_7d', 'forecast_30d', 'quiebres_7d', 'quiebres_30d', 'visitas_tienda_30d',
    'tickets_30d', 'inversion_trade', 'espacio_gondola_cm', 'lead_time_dias',
    'distancia_cd_km', 'capacidad_bodega_m3', 'vencimiento_dias', 'vida_util_dias'
]

for col in cols_no_negativas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df.loc[df[col] < 0, col] = np.nan

# 5.5 Duplicados por llave de negocio
# TODO: elimina duplicados usando sku_id, tienda_id y fecha.
# Pista: df = df.drop_duplicates(subset=['sku_id', 'tienda_id', DATE_COL])

print(df.shape)
df.head()



## 6. Feature engineering

Crea variables que conecten mejor con la decision de reposicion.


In [ ]:
# Features sugeridas
# Puedes completar o modificar estas formulas.

df['gap_stock_vs_forecast_7d'] = df['forecast_7d'] - df['stock_actual']
df['gap_stock_vs_forecast_30d'] = df['forecast_30d'] - df['stock_actual']
df['venta_promedio_diaria_30d'] = df['ventas_unidades_30d'] / 30

df['ratio_stock_seguridad'] = df['stock_actual'] / df['stock_seguridad'].replace(0, np.nan)
df['presion_demanda_7d'] = df['forecast_7d'] + df['quiebres_7d']
df['rentabilidad_unitaria'] = df['precio_venta'] - df['costo_unitario']

# TODO: crea al menos 3 features adicionales.
# Ideas:
# - flag_stock_bajo
# - flag_promocion_con_demanda
# - ratio_forecast_ventas
# - riesgo_vencimiento

# df['flag_stock_bajo'] = ...
# df['ratio_forecast_ventas'] = ...
# df['riesgo_vencimiento'] = ...



## 7. EDA guiado

Haz graficos para entender el comportamiento del target.


In [ ]:
# Distribucion del target
df[TARGET].value_counts().sort_index().plot(kind='bar', title='Distribucion del target')
plt.xlabel('target_reponer_sku')
plt.ylabel('Registros')
plt.show()

# Tasa de target por categoria
(df.groupby('categoria')[TARGET]
   .mean()
   .sort_values(ascending=False)
   .plot(kind='barh', title='Tasa promedio de reposicion por categoria'))
plt.xlabel('Promedio target')
plt.show()


In [ ]:
# TODO: crea 3 graficos adicionales.
# Sugerencias:
# 1. Boxplot de stock_actual por target.
# 2. Boxplot de forecast_7d por target.
# 3. Tasa de target por promo_activa.
# 4. Tasa de target por prioridad_comercial.
# 5. Scatter stock_actual vs forecast_7d coloreado por target.

# Escribe tu codigo debajo:



## 8. Separacion train/test

No se debe evaluar con los mismos datos usados para entrenar. Usamos `stratify=y` para mantener proporciones del target.


In [ ]:
# Excluimos IDs puros y target.
# Nota: en algunos casos sku_id y tienda_id pueden ser utiles como historicos agregados,
# pero como identificadores directos pueden sobreajustar.

MODEL_SAMPLE_SIZE = 3000  # Para que el taller corra rapido. Cambia a len(df) para usar todo.
df_model = df.sample(n=min(MODEL_SAMPLE_SIZE, len(df)), random_state=42).copy()

features_to_drop = [TARGET, 'sku_id', 'tienda_id']
X = df_model.drop(columns=[c for c in features_to_drop if c in df_model.columns])
y = df_model[TARGET]

# Convertimos fecha a features simples si existe.
if DATE_COL in X.columns:
    X[DATE_COL] = pd.to_datetime(X[DATE_COL], errors='coerce')
    X['fecha_mes'] = X[DATE_COL].dt.month
    X['fecha_dia_semana'] = X[DATE_COL].dt.dayofweek
    X = X.drop(columns=[DATE_COL])

# TODO: cambia test_size si quieres experimentar.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)



## 9. Preprocesamiento con Pipeline

Usamos `ColumnTransformer` para imputar, escalar numericas y codificar categoricas.


In [ ]:
numeric_features = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number', 'bool']).columns.tolist()

print('Numericas:', len(numeric_features))
print('Categoricas:', len(categorical_features))

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])



## 10. Modelos baseline

Primero entrenamos modelos simples para tener una linea base.


In [ ]:
def evaluate_model(name, model, X_test, y_test, threshold=0.5):
    # Evalua un modelo binario con probabilidades.
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        'modelo': name,
        'threshold': threshold,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, proba),
        'pr_auc': average_precision_score(y_test, proba)
    }


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced')
}

# TODO avanzado: agrega modelos mas potentes para comparar.
# models['Random Forest'] = RandomForestClassifier(n_estimators=35, max_depth=7, random_state=42, class_weight='balanced', n_jobs=-1)
# models['Gradient Boosting'] = GradientBoostingClassifier(n_estimators=40, random_state=42)

results = []
fitted_models = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', clf)])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    results.append(evaluate_model(name, pipe, X_test, y_test, threshold=0.5))

pd.DataFrame(results).sort_values('f1', ascending=False)



## 11. Ensemble models

Un ensemble combina varios modelos. Dos enfoques comunes:

- **VotingClassifier:** combina votos o probabilidades.
- **StackingClassifier:** usa modelos base y un meta-modelo que aprende a combinarlos.

**Actividad:** entrena ambos y compara con los modelos base.


In [ ]:
# TODO avanzado: activa este bloque cuando ya tengas claro el baseline.
# Voting soft promedia probabilidades de modelos base.
#
# voting_model = VotingClassifier(
#     estimators=[
#         ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
#         ('rf', RandomForestClassifier(n_estimators=30, max_depth=7, random_state=42, class_weight='balanced', n_jobs=-1)),
#         ('gb', GradientBoostingClassifier(n_estimators=40, random_state=42))
#     ],
#     voting='soft'
# )
#
# voting_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', voting_model)])
# voting_pipe.fit(X_train, y_train)
#
# ensemble_results = [evaluate_model('Voting Ensemble', voting_pipe, X_test, y_test)]
# pd.DataFrame(ensemble_results)


In [ ]:
# TODO: completa un StackingClassifier.
# Pista:
# stacking_model = StackingClassifier(
#     estimators=[...],
#     final_estimator=LogisticRegression(max_iter=1000),
#     stack_method='predict_proba'
# )
# stacking_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', stacking_model)])
# stacking_pipe.fit(X_train, y_train)
# evaluate_model('Stacking Ensemble', stacking_pipe, X_test, y_test)




## 12. Elegir umbral de decision

El modelo entrega probabilidades. El umbral define cuando recomendamos reponer.

- Umbral bajo: mas recall, mas recomendaciones, riesgo de sobrestock.
- Umbral alto: mas precision, menos recomendaciones, riesgo de no detectar quiebres.


In [ ]:
best_model_name = 'Logistic Regression'  # TODO: cambia por el mejor modelo segun tus metricas.
best_model = fitted_models[best_model_name]

proba_test = best_model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.05)
threshold_results = []
for th in thresholds:
    pred = (proba_test >= th).astype(int)
    threshold_results.append({
        'threshold': round(th, 2),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'recomendaciones_pct': pred.mean()
    })

threshold_df = pd.DataFrame(threshold_results)
threshold_df.sort_values('f1', ascending=False).head(10)


In [ ]:
# TODO: grafica precision, recall y f1 por threshold.
# Pista:
# threshold_df.plot(x='threshold', y=['precision','recall','f1'])
# plt.show()



## 13. Matriz de confusion y reporte final

Elige un umbral segun el objetivo de negocio.


In [ ]:
UMBRAL_FINAL = 0.50  # TODO: ajusta segun el trade-off precision/recall.

pred_final = (proba_test >= UMBRAL_FINAL).astype(int)
print(confusion_matrix(y_test, pred_final))
print(classification_report(y_test, pred_final))


In [ ]:
# Curvas ROC y Precision-Recall
RocCurveDisplay.from_predictions(y_test, proba_test)
plt.title(f'ROC Curve - {best_model_name}')
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, proba_test)
plt.title(f'Precision-Recall Curve - {best_model_name}')
plt.show()



## 14. Explicabilidad

Para negocio necesitamos explicar por que el modelo recomienda reponer. Una forma simple es usar permutation importance.


In [ ]:
# Para ahorrar tiempo, calculamos permutation importance en una muestra del test.
sample_size = min(120, len(X_test))
X_sample = X_test.sample(sample_size, random_state=42)
y_sample = y_test.loc[X_sample.index]

perm = permutation_importance(
    best_model, X_sample, y_sample,
    n_repeats=2, random_state=42, scoring='f1'
)

importance_df = pd.DataFrame({
    'feature': X_sample.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

importance_df.head(15)


In [ ]:
# TODO: interpreta las 5 variables mas importantes.
# Escribe en comentarios o Markdown:
# 1.
# 2.
# 3.
# 4.
# 5.



## 15. Prediccion para nuevos casos

Simulamos como se usaria el modelo en negocio. La salida debe incluir probabilidad, decision y accion sugerida.


In [ ]:
# Tomamos algunos registros del test como si fueran nuevos casos.
nuevos_casos = X_test.head(10).copy()
probabilidades = best_model.predict_proba(nuevos_casos)[:, 1]
decisiones = (probabilidades >= UMBRAL_FINAL).astype(int)

salida = nuevos_casos.copy()
salida['prob_reponer'] = probabilidades.round(3)
salida['decision_reponer'] = decisiones

# Agregamos IDs desde df original usando indices si estan disponibles.
salida[['prob_reponer', 'decision_reponer']].head(10)


In [ ]:
# TODO: crea una columna accion_sugerida.
# Regla ejemplo:
# - Si prob_reponer >= 0.75: 'Reposicion urgente'
# - Si prob_reponer >= 0.50: 'Reponer en siguiente ciclo'
# - Si prob_reponer < 0.50: 'Monitorear'

# salida['accion_sugerida'] = ...
# salida[['prob_reponer', 'decision_reponer', 'accion_sugerida']].head(10)



## 16. Beneficio de negocio

La ciencia de datos debe traducirse en impacto. Ejemplos de beneficios:

- Menos quiebres de stock.
- Menos ventas perdidas.
- Mejor fill rate.
- Menor sobrestock y merma.
- Mejor priorizacion del transporte y abastecimiento.

**Actividad:** crea una simulacion simple de beneficio.


In [ ]:
# Simulacion simple de impacto
# Supuestos didacticos, debes ajustarlos al negocio real.
venta_promedio_por_sku = 80       # soles por oportunidad recuperada
costo_sobrestock_por_sku = 15     # soles por falso positivo

cm = confusion_matrix(y_test, pred_final)
tn, fp, fn, tp = cm.ravel()

beneficio_recuperado = tp * venta_promedio_por_sku
costo_sobrestock = fp * costo_sobrestock_por_sku
beneficio_neto = beneficio_recuperado - costo_sobrestock

print('TP:', tp, 'FP:', fp, 'FN:', fn, 'TN:', tn)
print('Beneficio recuperado estimado:', beneficio_recuperado)
print('Costo sobrestock estimado:', costo_sobrestock)
print('Beneficio neto estimado:', beneficio_neto)

# TODO: cambia los supuestos y analiza como cambia el beneficio.



## 17. Conclusiones del proyecto

Completa tus conclusiones:

1. Que modelo eliges y por que?
2. Que umbral recomiendas?
3. Que variables explican mejor la decision?
4. Que beneficio de negocio esperas?
5. Que datos adicionales pedirias para mejorar el modelo?
6. Que riesgos existen antes de llevarlo a produccion?


In [ ]:
# TODO: escribe tus conclusiones finales aqui o en una celda Markdown.



## 18. Checklist final de ciencia de datos

Marca lo completado:

- [ ] Entendi la decision de negocio.
- [ ] Identifique el target y su significado.
- [ ] Revise calidad de datos.
- [ ] Limpie tipos, nulos, duplicados y outliers.
- [ ] Cree nuevas features.
- [ ] Hice EDA con preguntas de negocio.
- [ ] Separe train/test correctamente.
- [ ] Entrene baseline y modelos mas avanzados.
- [ ] Compare metricas.
- [ ] Ajuste el threshold.
- [ ] Explique variables importantes.
- [ ] Genere predicciones accionables.
- [ ] Traduci el modelo a beneficio de negocio.
